# v17: Level probes on the v12_robust base (budget size, number of flags)

**Version:** v17  
**Kaggle submission:** `v17_bud125` (#16) and `v17_k135` (#17) → `outputs/submission_v17_*.csv` (each a single change vs `v12_robust`)

**Question:** ranking changes (seed averaging, sensor mapping) moved the public score by a few hundred points in both directions, inside the noise. Level changes (the shift k) moved it by thousands. Two levels are still unprobed on the best base: are flagged budgets too small (the test depot's repairs may be more expensive than training's), and do we flag too few vehicles? One file per question, each a single change vs `v12_robust`:

- **`v17_bud125`:** same flags, every flagged budget ×1.25.
- **`v17_k135`:** shift factor k scaled by 1.35 (≈ 7% more flags), budgets from the same newsvendor rule.

*Self-contained: needs only `../data/train.csv` and `../data/test.csv`. Part of the Fleet Maintenance project (Kaggle `mlprojectfleet2`); the cost, decision rule and validation protocol are derived in `fleet_maintenance.ipynb`.*

In [1]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, xgboost as xgb
from pathlib import Path
from scipy.stats import ks_2samp
from sklearn.isotonic import IsotonicRegression
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, average_precision_score, r2_score, brier_score_loss, log_loss
from sklearn.model_selection import StratifiedKFold, KFold, train_test_split, cross_val_predict

DATA, OUT = Path("../data"), Path("outputs"); OUT.mkdir(exist_ok=True)
SEED = 42
INSPECT, UNDER, OVER = 10.0, 0.20, 0.02          # inspection cost; shortage cost / unit; unused-budget cost / unit
Q_STAR = UNDER / (UNDER + OVER)                  # newsvendor critical ratio 0.909
HISTB = [f"s{i:03d}" for i in range(54, 64)]     # 10-bin histogram that is shifted by one bin in test

def load(raw=False):
    """train/test with the v1 fixes: realign s054-s063 (test bin k+1 == train bin k), drop s047 (all NaN in
    test) and s146 (constant). raw=True returns the files untouched."""
    tr, te = pd.read_csv(DATA / "train.csv"), pd.read_csv(DATA / "test.csv")
    if not raw:
        te[HISTB[:-1]] = te[HISTB[1:]].to_numpy(); te[HISTB[-1]] = np.nan
        tr, te = tr.drop(columns=["s047", "s146"]), te.drop(columns=["s047", "s146"])
    return tr, te, [c for c in tr.columns if c.startswith("s")]

tr, te, S = load()
dev_idx, ho_idx = train_test_split(np.arange(len(tr)), test_size=0.2, stratify=tr.failure, random_state=SEED)
dev, ho = tr.iloc[dev_idx].reset_index(drop=True), tr.iloc[ho_idx].reset_index(drop=True)   # ho: locked 20% hold-out
y, c = dev.failure.to_numpy(), dev.repair_cost.fillna(0).to_numpy() * dev.failure.to_numpy()
y_ho, c_ho = ho.failure.to_numpy(), ho.repair_cost.fillna(0).to_numpy() * ho.failure.to_numpy()
c_all = tr.repair_cost.fillna(0).to_numpy() * tr.failure.to_numpy()
F = list(StratifiedKFold(5, shuffle=True, random_state=SEED).split(np.zeros(len(y)), y))   # the 5 fixed folds
print(f"train {tr.shape}  test {te.shape}  sensors {len(S)}  dev {len(dev)} / hold-out {len(ho)}  failures {int(y.sum())}")

train (56000, 172)  test (20000, 169)  sensors 168  dev 44800 / hold-out 11200  failures 318


In [2]:
def vehicle_cost(c, flag, b):
    return np.where(flag == 1, INSPECT + UNDER * np.maximum(c - b, 0) + OVER * np.maximum(b - c, 0), c)

def per_vehicle(c, flag, b):
    return float(vehicle_cost(np.asarray(c, float), np.asarray(flag), np.asarray(b, float)).mean())

fold_cost = lambda flag, b: np.array([per_vehicle(c[v], flag[v], b[v]) for _, v in F])

def write_submission(ids, flag, budget, name):
    flag, budget = np.asarray(flag).astype(int), np.maximum(np.asarray(budget, float), 0)
    assert len(ids) == 20000 and ids.is_unique and set(flag) <= {0, 1} and np.isfinite(budget).all()
    pd.DataFrame({"id": ids, "flag": flag, "budget": budget.round(2)}).to_csv(OUT / f"submission_{name}.csv", index=False)
    print(f"wrote outputs/submission_{name}.csv: flagged {flag.mean():.2%} of test vehicles")

def decide_global(p, fail_costs):
    """Same repair-cost distribution F (empirical failure costs) for every vehicle."""
    p = np.clip(np.asarray(p, float), 0, 1); fc = np.sort(np.asarray(fail_costs, float)); mu = fc.mean()
    b = np.zeros_like(p); need = p > 1 - Q_STAR
    b[need] = np.quantile(fc, 1 - (1 - Q_STAR) / p[need])                       # newsvendor budget
    fail = (UNDER * np.maximum(fc[None, :] - b[:, None], 0) + OVER * np.maximum(b[:, None] - fc[None, :], 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b   # flag iff cheaper than not flagging

def decide_pv(p, m, resid, fail_costs=None, K=99, scale=None):
    """Per-vehicle cost distribution: log c | fail ~ m_i + scale_i * residuals (out-of-fold residuals).
    fail_costs given -> budget from that GLOBAL distribution (ablation); scale -> per-vehicle spread (v9)."""
    p, m = np.clip(np.asarray(p, float), 0, 1), np.asarray(m, float)
    sc = np.ones_like(m) if scale is None else np.asarray(scale, float)
    M = np.exp(m[:, None] + sc[:, None] * np.quantile(resid, (np.arange(K) + 0.5) / K)[None, :]); mu = M.mean(1)
    b = np.zeros_like(p); need = p > 1 - Q_STAR; level = 1 - (1 - Q_STAR) / p[need]
    b[need] = (np.exp(m[need] + sc[need] * np.quantile(resid, level)) if fail_costs is None
               else np.quantile(np.asarray(fail_costs, float), level))
    fail = (UNDER * np.maximum(M - b[:, None], 0) + OVER * np.maximum(b[:, None] - M, 0)).mean(1)
    return ((INSPECT + p * fail + (1 - p) * OVER * b) < p * mu).astype(int), b

class Platt:
    def fit(self, p, y):
        self.m = LogisticRegression(C=1e6, max_iter=1000).fit(self._x(p), y); return self
    def predict(self, p): return self.m.predict_proba(self._x(p))[:, 1]
    @staticmethod
    def _x(p): p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))[:, None]

class Iso:
    def fit(self, p, y):
        self.m = IsotonicRegression(out_of_bounds="clip", y_min=0, y_max=1).fit(p, y); return self
    def predict(self, p): return self.m.predict(p)

make_cal = lambda kind: {"platt": Platt, "iso": Iso}[kind]()

PARAMS = dict(n_estimators=300, max_depth=4, learning_rate=0.05, subsample=0.8, colsample_bytree=0.8,
              min_child_weight=1, n_jobs=-1, random_state=SEED)                         # failure classifier (v2..)
CM = dict(n_estimators=200, max_depth=3, learning_rate=0.03, subsample=0.8, colsample_bytree=0.5,
          min_child_weight=5, n_jobs=-1, random_state=SEED)                             # repair-cost model (v5)
inner = KFold(5, shuffle=True, random_state=SEED)

def cost_model(params, Xa, lc, Xb):
    """XGBoost on log repair cost (failures only). Returns predictions for Xb and OUT-OF-FOLD residuals of Xa."""
    resid = lc - cross_val_predict(xgb.XGBRegressor(**params), Xa, lc, cv=inner)
    return xgb.XGBRegressor(**params).fit(Xa, lc).predict(Xb), resid

def fit_oof(X, params=PARAMS):
    """5-fold out-of-fold failure probabilities on dev, and probabilities for test from a model fit on all train."""
    o = np.zeros(len(dev))
    for a, b in F:
        o[b] = xgb.XGBClassifier(**params).fit(X.iloc[a], y[a]).predict_proba(X.iloc[b])[:, 1]
    return o

shift = lambda p, k: k * p / (1 - p) / (1 + k * p / (1 - p))       # prior shift by odds factor k

In [3]:
HIST = {a: [f"s{i:03d}" for i in range(a, a + 10)] for a in (23, 37, 54, 80, 129, 154)}   # six hidden 10-bin histograms
HCOLS = {col for g in HIST.values() for col in g}

def shape(df):
    """Histogram-shape features: per-bin shares, mean/std bin, entropy, low/high-3 tail shares, log usage."""
    out = {"logU": np.log1p(df[HIST[54][:9]].sum(axis=1, min_count=9))}
    for a, g in HIST.items():
        g = g[:9] if a == 54 else g                        # test s063 is NaN after realignment
        x = df[g].to_numpy(float); tot = x.sum(1, keepdims=True)
        sh = np.where(tot > 0, x / np.where(tot > 0, tot, 1), np.nan); k = np.arange(len(g)); mean = (sh * k).sum(1)
        out.update({f"h{a}_sh{i}": sh[:, i] for i in range(len(g))})
        out[f"h{a}_mean"] = np.where(tot[:, 0] > 0, mean, np.nan)
        out[f"h{a}_std"] = np.where(tot[:, 0] > 0, np.sqrt(np.maximum((sh * k ** 2).sum(1) - mean ** 2, 0)), np.nan)
        with np.errstate(divide="ignore", invalid="ignore"):
            out[f"h{a}_ent"] = np.where(tot[:, 0] > 0, -np.nansum(np.where(sh > 0, sh * np.log(sh), 0), 1), np.nan)
        out[f"h{a}_lo3"], out[f"h{a}_hi3"] = sh[:, :3].sum(1), sh[:, -3:].sum(1)
    return pd.DataFrame(out, index=df.index)

def ratios(df, cols):
    U = df[HIST[54][:9]].sum(axis=1, min_count=9).replace(0, np.nan)
    return df[[col for col in cols if col not in HCOLS]].div(U, axis=0).add_suffix("_perU")

In [4]:
CM9 = {**CM, "n_estimators": 1000, "max_depth": 2}                      # v9 cost model
DRIFT = ["s120", "s122", "s126", "s128"]; MAP = ["s120", "s122"]
keep = [s for s in S if s not in DRIFT]
f_all = tr[tr.failure == 1]; lc_all = np.log(f_all.repair_cost.to_numpy())
Xf, Xt = pd.concat([f_all[keep], shape(f_all)], axis=1), pd.concat([te[keep], shape(te)], axis=1)   # cost model: no drifted sensors (v12)
# posterior over the depot shift k from the v10 notebook (nine public scores), full precision
GRID = [1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0, 5.5, 6.0, 7.0, 8.0, 10.0, 12.0]
W = [0.008172761879954695, 0.13116481847950048, 0.26721015776942314, 0.267056932957599, 0.1710924552535038, 0.0956936236265262,
     0.03459275261241694, 0.016549303552639335, 0.005725443007721319, 0.002368204370941116, 0.00031775367779660887,
     5.437858042612226e-05, 1.3770078232241008e-06, 3.722372808678639e-08]
pavg = lambda qq, lam=1.0: sum(w * shift(qq, k * lam) for w, k in zip(W, GRID))   # posterior-averaged shifted probability

### 1. Reproduce `v12_robust` (single-seed classifier, cost model without the drifted sensors)

In [5]:
q = Platt().fit(fit_oof(dev[S]), y).predict(xgb.XGBClassifier(**PARAMS).fit(tr[S], tr.failure).predict_proba(te[S])[:, 1])
m, resid = cost_model(CM9, Xf, lc_all, Xt)
p0 = pavg(q); fl12, bu12 = decide_pv(p0, m, resid)
r12 = pd.read_csv("../submissions/submission_v12_robust.csv")
assert (r12.flag.to_numpy() == fl12).all() and np.allclose(r12.budget, bu12.round(2), atol=0.011), "v12_robust not reproduced"
print(f"v12_robust reproduced: {int(fl12.sum())} flags")

v12_robust reproduced: 1761 flags


### 2. The two probes

In [6]:
bu_a = np.where(fl12 == 1, bu12 * 1.25, bu12)
write_submission(te.id, fl12, bu_a, "v17_bud125")
fl_b, bu_b = decide_pv(pavg(q, 1.35), m, resid)
write_submission(te.id, fl_b, bu_b, "v17_k135")
for nm, fl, bu in (("v17_bud125", fl12, bu_a), ("v17_k135", fl_b, bu_b)):
    ref = pd.read_csv(f"../submissions/submission_{nm}.csv")
    assert (ref.flag.to_numpy() == fl).all() and np.allclose(ref.budget, bu.round(2), atol=0.011), f"{nm} not reproduced"
print(f"v17_k135: {int(fl_b.sum())} flags ({int(fl_b.sum() - fl12.sum()):+d} vs v12_robust, all added, none removed: "
      f"{bool((fl_b >= fl12).all())})")

wrote outputs/submission_v17_bud125.csv: flagged 8.80% of test vehicles
wrote outputs/submission_v17_k135.csv: flagged 9.40% of test vehicles
v17_k135: 1880 flags (+119 vs v12_robust, all added, none removed: True)


### 3. What did our own model expect these changes to do?

Before reading the public scores: under the model (posterior-averaged p, per-vehicle cost distribution) we compute each file's expected total cost on a random 30% of test, the size of the public subset. If the model already says a change is worth only a few dozen points, a public result within noise is what we should expect, and it says nothing against the model.

In [7]:
def expected_cost(p, m, resid, flag, b, K=99):
    """Model-expected cost per vehicle: p = failure probability, log c | fail ~ m + out-of-fold residual quantiles."""
    M = np.exp(m[:, None] + np.quantile(resid, (np.arange(K) + 0.5) / K)[None, :])
    fail = (UNDER * np.maximum(M - b[:, None], 0) + OVER * np.maximum(b[:, None] - M, 0)).mean(1)
    return np.where(flag == 1, INSPECT + p * fail + (1 - p) * OVER * b, p * M.mean(1))

base = expected_cost(p0, m, resid, fl12, bu12)
for nm, fl, bu in (("v17_bud125", fl12, bu_a), ("v17_k135", fl_b, bu_b)):
    d = expected_cost(p0, m, resid, fl, bu) - base
    print(f"{nm}: model-expected change on the public 30%: {0.3 * d.sum():+8.1f}")
for k in (2.0, 3.0, 4.0):                                       # if the true shift were one fixed k instead of the posterior
    pk = shift(q, k); b0 = expected_cost(pk, m, resid, fl12, bu12)
    print(f"  at k={k:.0f}: bud125 {0.3 * (expected_cost(pk, m, resid, fl12, bu_a) - b0).sum():+7.1f}   "
          f"k135 {0.3 * (expected_cost(pk, m, resid, fl_b, bu_b) - b0).sum():+7.1f}")

v17_bud125: model-expected change on the public 30%:   +275.2
v17_k135: model-expected change on the public 30%:   +117.4
  at k=2: bud125  +434.2   k135  +380.4
  at k=3: bud125  +267.2   k135  +114.7


  at k=4: bud125  +138.2   k135  -129.3


## Results and decisions

| file | change vs v12_robust | public score | difference |
|---|---|---|---|
| `v12_robust` | n/a | 16,839.86 | n/a |
| `v17_bud125` | flagged budgets ×1.25 | 16,826 | −14 |
| `v17_k135` | k × 1.35 (≈ 7% more flags) | 16,908 | +68 |

- **Both are ties.** The differences (−14 and +68) are far inside the ≈ 600-point noise between sibling files.
- **The model predicted small changes, both slightly worse:** +275 for bud125 and +117 for k135 on the public 30% (posterior over k). k135 landed where predicted (+68). bud125 came out ≈ 290 better than predicted: a hint, well inside noise, that test repairs cost a bit more than the training-fitted cost model expects.
- **Budget size is not a lever we can read.** Only ~1% of vehicles fail, so only the few flagged *failures* feel the budget; over-budgeting costs 0.02 per unit on every flagged vehicle, under-budgeting 0.20 on the failures, and near the newsvendor quantile the two roughly cancel. We keep the newsvendor budget.
- **Flag count is at its optimum.** The posterior over k came from nine earlier public scores; ≈ 7% more flags on top of it gave the small loss the model expected, consistent with sitting near the flat bottom of the cost-vs-k curve.
- **Conclusion:** every lever that the level (k, budget size) controls has now been used. The remaining gap to the leader can only come from *ranking*: which vehicles get flagged, i.e. the classifier's quality at the new depot.

**Caveats:** one public comparison per probe; the 70% private part could react differently, but neither probe gives a reason to prefer it over `v12_robust`.